In [1]:
!pip install stable-baselines3[extra] pettingzoo[atari] supersuit autorom[accept-rom-license]
!AutoROM --accept-license


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 13.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 707.8/707.8 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.6/563.6 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.5/805.5 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 12.8 MB/s eta 0:00:00
  Created wheel for AutoROM.accept-rom-license: filename=autorom_accept_rom_license-0.6.1-py3-none-any.whl size=446710 sha256=da1424b7def7900ad875111f10aab88817830e71d39bbb7c503656b98cd1db91
  Stored in directory: /root/.c

In [2]:
import numpy as np
import supersuit as ss
from pettingzoo.atari import warlords_v3
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecMonitor
from stable_baselines3.common.callbacks import CheckpointCallback

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
def make_env():
    env = warlords_v3.parallel_env(obs_type="ram", render_mode=None)
    env = ss.black_death_v3(env)
    env = ss.pettingzoo_env_to_vec_env_v1(env)
    env = ss.concat_vec_envs_v1(env, num_vec_envs=1, num_cpus=1, base_class="stable_baselines3")
    return env

env = make_env()
env = VecMonitor(env)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

Observation space: Box(0, 255, (128,), uint8)
Action space: Discrete(6)


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/base_vec_env.py:78: UserWarning: The `render_mode` attribute is not defined in your environment. It will be set to None.
  warnings.warn("The `render_mode` attribute is not defined in your environment. It will be set to None.")


In [4]:
model = PPO(
    policy="MlpPolicy",
    env=env,
    verbose=1,
    n_steps=512,
    batch_size=256,
    n_epochs=4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    learning_rate=2.5e-4,
    device="cuda",
)

Using cpu device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
checkpoint_cb = CheckpointCallback(save_freq=100_000, save_path="./checkpoints/", name_prefix="ppo_warlords")

model.learn(total_timesteps=2_000_000, callback=checkpoint_cb, progress_bar=True)
model.save("ppo_warlords_final")
print("Done!")

Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

Streaming output truncated to the last 5000 lines.
|    value_loss           | 3.76e-06     |
------------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 6.6e+03     |
|    ep_rew_mean          | -0.5        |
| time/                   |             |
|    fps                  | 1791        |
|    iterations           | 733         |
|    time_elapsed         | 838         |
|    total_timesteps      | 1501184     |
| train/                  |             |
|    approx_kl            | 0.001041604 |
|    clip_fraction        | 0.00134     |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.75       |
|    explained_variance   | 0.961       |
|    learning_rate        | 0.00025     |
|    loss                 | -0.0214     |
|    n_updates            | 2928        |
|    policy_gradient_loss | -0.000947   |
|    value_loss           | 1.49e-06    |
-----------------------

Done!


In [8]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copy('ppo_warlords_final.zip', '/content/drive/MyDrive/ppo_warlords_final.zip')
print("Saved to Google Drive!")

Mounted at /content/drive
Saved to Google Drive!
